# Day 03 · MCP / FastMCP 실습
FastMCP로 **Tool·Resource·Prompt**를 만들고, 노트북 안에서 **in-process Client**로 직접 호출해 검증합니다.

> 노트북에서는 `fastmcp.Client`로 서버 객체에 바로 붙어 테스트할 수 있어 별도 프로세스가 필요 없습니다. (실제 배포 시엔 `mcp.run()` 또는 `fastmcp dev inspector` 사용)


## 0 · 설치


In [ ]:
!pip install -q fastmcp


In [ ]:
import fastmcp
print("fastmcp", fastmcp.__version__)


## 데코레이터 문법 배우기 — `@mcp.tool`을 이해하는 최소 파이썬
MCP 서버는 `@mcp.tool`·`@mcp.resource`·`@mcp.prompt` **데코레이터**로 만듭니다. 그래서 서버를 만들기 전에 데코레이터 문법을 단계로 익힙니다.
① 함수는 값이다 → ② 함수를 감싸는 함수 → ③ `@` 문법 → ④ `functools.wraps` → ⑤ 인자를 받는 데코레이터 → ⑥ `@mcp.tool`의 정체.

In [ ]:
# ① 함수도 '값(객체)'이다 — 변수에 담고, 넘기고, 돌려줄 수 있다
def hello():
    return "안녕"

f = hello                 # 괄호 없이 = 호출이 아니라 '함수 자체'를 담기
print(f())                # 안녕  (담아둔 함수를 호출)
print(type(hello), hello.__name__)   # 함수도 타입·이름을 가진 객체

In [ ]:
# ② 데코레이터 = 함수를 받아, '감싼' 새 함수를 돌려주는 함수
def announce(func):
    def wrapper(*args, **kwargs):     # 원래 함수의 어떤 인자든 그대로 받는다
        print(f"[호출] {func.__name__}")
        return func(*args, **kwargs)  # 원래 함수 실행
    return wrapper                    # 감싼 함수를 돌려준다

def greet(name):
    return f"안녕, {name}"

greet = announce(greet)   # 수동으로 감싸기
print(greet("민수"))

In [ ]:
# ③ @announce 는 위의 'greet = announce(greet)' 한 줄을 대신하는 문법(설탕)
@announce
def greet(name):
    return f"안녕, {name}"

print(greet("서연"))

# *args, **kwargs 덕분에 인자 개수가 달라도 그대로 감싼다
@announce
def add(a, b, c):
    return a + b + c
print(add(1, 2, 3))

In [ ]:
# ④ functools.wraps — 감싸도 원래 함수의 이름·docstring을 유지
import functools

def announce(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@announce
def greet(name):
    "이름을 받아 인사한다"
    return f"안녕, {name}"

print(greet.__name__)   # greet   (wraps 없으면 'wrapper' 로 뭉개진다)
print(greet.__doc__)    # 이름을 받아 인사한다
# → MCP는 함수 이름을 '도구명', docstring을 '설명'으로 쓰므로 이 유지가 중요

In [ ]:
# ⑤ 인자를 받는 데코레이터: @repeat(3) 처럼 ()가 붙으면 '데코레이터를 만드는 함수'다
def repeat(n):                         # 1) 설정 n 을 받아서
    def deco(func):                    # 2) 데코레이터를 돌려주고
        @functools.wraps(func)
        def wrapper(*args, **kwargs):  # 3) 그 데코레이터가 함수를 감싼다
            for _ in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return deco

@repeat(3)              # repeat(3) 이 먼저 실행돼 데코레이터를 만든 뒤 hi 에 적용
def hi():
    print("안녕")
hi()
# → @mcp.tool(annotations={"readOnlyHint": True}) 이 바로 이 형태 (인자 받는 데코레이터)

In [ ]:
# ⑥ @mcp.tool 의 정체: 함수를 목록에 '등록'하고 그대로 돌려준다
TOOLS = {}
def tool(func):
    TOOLS[func.__name__] = func
    return func

@tool
def add(a, b): return a + b
@tool
def shout(s): return s.upper() + "!"

print("등록된 도구:", list(TOOLS), "| 호출:", TOOLS["add"](2, 3))
# @mcp.tool 은 TOOLS 대신 'MCP 서버'에 등록할 뿐. 이제 Lab 1의 @mcp.tool 이 자연스럽게 읽힌다.

# TODO: 위 repeat 를 참고해, 함수를 두 번 실행하는 @twice 데코레이터를 직접 만들어 보세요

## Lab 1 · 첫 서버 + Tool 두 개
`@mcp.tool` — 부작용/계산 같은 '행동'. 타입힌트·docstring이 스키마·설명이 됩니다.


In [ ]:
from fastmcp import FastMCP

mcp = FastMCP("MyFirstServer")

@mcp.tool
def add(a: int, b: int) -> int:
    """두 정수를 더한다"""
    return a + b

@mcp.tool
def word_count(text: str) -> int:
    """문장의 단어 수를 센다"""
    return len(text.split())

print("도구 등록 완료")


## Lab 2 · in-process Client로 호출
노트북은 top-level `await`를 지원합니다. 도구 목록을 보고 직접 실행해 봅니다.


In [ ]:
from fastmcp import Client

async with Client(mcp) as client:
    tools = await client.list_tools()
    print("도구 목록:", [t.name for t in tools])
    r1 = await client.call_tool("add", {"a": 2, "b": 3})
    r2 = await client.call_tool("word_count", {"text": "MCP 서버 정말 쉽다"})
    print("add(2,3) =", r1.data)
    print("word_count =", r2.data)


## Lab 3 · Resource 추가 (읽기 전용)
`@mcp.resource` — GET 같은 읽기. URI로 주소를 지정하고 **description을 꼭 명시**합니다.


In [ ]:
STATUS = {"version": "1.0", "env": "dev"}

@mcp.resource("config://status", description="서버 상태 정보")
def status() -> dict:
    return STATUS

async with Client(mcp) as client:
    res = await client.list_resources()
    print("리소스:", [str(r.uri) for r in res])
    data = await client.read_resource("config://status")
    print("config://status →", data[0].text)


## Lab 4 · Prompt 추가 (재사용 템플릿)
`@mcp.prompt` — 대화를 세팅하는 템플릿. 슬래시 명령처럼 노출됩니다.


In [ ]:
@mcp.prompt
def code_review(language: str, code: str) -> str:
    """코드 리뷰 요청 프롬프트"""
    return f"다음 {language} 코드를 리뷰하고 개선점을 알려줘:\n\n{code}"

async with Client(mcp) as client:
    prompts = await client.list_prompts()
    print("프롬프트:", [p.name for p in prompts])
    msg = await client.get_prompt("code_review", {"language": "python", "code": "print(1)"})
    print(msg.messages[0].content.text[:80], "...")


## Lab 5 · 실전 도구 — 메모 저장/검색
부작용(저장)은 Tool. add_note로 저장 → search_notes로 검색.


In [ ]:
NOTES = []

@mcp.tool
def add_note(text: str) -> str:
    """메모를 추가한다"""
    NOTES.append(text)
    return f"저장됨 (총 {len(NOTES)}개)"

@mcp.tool
def search_notes(keyword: str) -> list:
    """키워드가 든 메모를 찾는다"""
    return [n for n in NOTES if keyword in n]

async with Client(mcp) as client:
    await client.call_tool("add_note", {"text": "내일 회의 10시"})
    await client.call_tool("add_note", {"text": "비용 정산 영수증 첨부"})
    found = await client.call_tool("search_notes", {"keyword": "회의"})
    print("검색 결과:", found.data)


## Lab 6 · RAG를 MCP 도구로
지난 시간(Day 01~02)의 검색을 MCP 도구로 감싸면, 어떤 AI 앱이든 `search_docs`로 부를 수 있습니다.
여기선 가볍게 키워드 검색을 쓰고, **실제로는 이 자리에 Day02의 하이브리드+리랭커(`advanced_rag`)를 넣습니다.**

확인: Client로 `search_docs`를 호출하면 상위 문서가 돌아오는가? (Day05에서는 이 도구를 LLM이 스스로 부릅니다.)

In [ ]:
DOCS = [
    "연차 휴가는 사용 3영업일 전까지 사내포털에서 신청하고 팀장 승인을 받는다.",
    "비용 정산은 영수증을 첨부해 정산 메뉴에서 접수하고 재무팀이 최종 승인한다.",
    "재택근무는 주 2회까지 가능하며 전날 18시까지 팀 채널에 공유한다.",
    "에러코드 ERR-4021은 인증 토큰 만료를 의미하며 재로그인으로 해결한다.",
]

@mcp.tool(annotations={"readOnlyHint": True})
def search_docs(query: str, k: int = 2) -> list:
    """사내 문서에서 질문과 관련된 문서를 찾는다 (Day01~02 검색을 도구로)"""
    ranked = sorted(DOCS, key=lambda d: -sum(w in d for w in query.split()))
    return ranked[:k]

async with Client(mcp) as c:
    r = await c.call_tool("search_docs", {"query": "연차 신청 며칠 전", "k": 2})
    print("검색 결과:")
    for line in r.data:
        print(" -", line)

## Lab 7 · 도전 (선택)
- **A** `mcp.run()`을 `server.py`로 저장 → 터미널에서 `fastmcp dev inspector`로 브라우저 검증
- **B** `mcp.run(transport="streamable-http")`로 원격화
- **C** 잘못된 인자에 친절한 오류를 반환하도록 입력 검증 추가

```python
# server.py 로 저장해 실행하는 형태
if __name__ == "__main__":
    mcp.run()   # 기본 stdio
```

> 📝 **산출물**: Tool·Resource·Prompt를 모두 갖춘 동작 서버.
